# Notebook 04: Functional and Transcription Factor Enrichment Analysis

**Input**: 20 convergent SIEM genes from notebook 03
**Output**:
- GO/KEGG enrichment results
- ChEA 2022 TF enrichment results
- Tables for manuscript

**Tools**: gseapy (Enrichr wrapper), Enrichr web service
**Databases**: GO BP 2021, KEGG 2021 Human, ChEA 2022

In [ ]:
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import numpy as np
import os

OUTPUT_DIR = '../tables'
FIGURE_DIR = '../figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

# Load convergent genes
conv = pd.read_csv(f'{OUTPUT_DIR}/convergent_genes_RRA_GSE50838_20genes.csv', index_col=0)
gene_list = list(conv.index)
print(f'Enrichment analysis for {len(gene_list)} SIEM genes:')
print(gene_list)

## 4.1 GO Biological Process + KEGG Pathway Enrichment (Enrichr)

In [ ]:
# GO BP 2021 enrichment
go_enr = gp.enrichr(
    gene_list=gene_list,
    gene_sets='GO_Biological_Process_2021',
    organism='human',
    outdir=None
)

go_results = go_enr.results
go_results = go_results.sort_values('Adjusted P-value')

# KEGG 2021 enrichment
kegg_enr = gp.enrichr(
    gene_list=gene_list,
    gene_sets='KEGG_2021_Human',
    organism='human',
    outdir=None
)

kegg_results = kegg_enr.results
kegg_results = kegg_results.sort_values('Adjusted P-value')

# Summary
print('=== GO Biological Process (Top 10) ===')
print(go_results[['Term', 'Overlap', 'Adjusted P-value', 'Odds Ratio']].head(10).to_string(index=False))

print('\n=== KEGG Pathways (Top 10) ===')
print(kegg_results[['Term', 'Overlap', 'Adjusted P-value', 'Odds Ratio']].head(10).to_string(index=False))

# Save
go_results.to_csv(f'{OUTPUT_DIR}/GO_enrichment_SIEM_20genes.csv', index=False)
kegg_results.to_csv(f'{OUTPUT_DIR}/KEGG_enrichment_SIEM_20genes.csv', index=False)
print(f'\nSaved: GO_enrichment_SIEM_20genes.csv')
print(f'Saved: KEGG_enrichment_SIEM_20genes.csv')

## 4.2 Transcription Factor Enrichment (ChEA 2022 via Enrichr)

**Caveat**: ChEA 2022 data are derived predominantly from ChIP-seq experiments in cancer cell lines (MCF-7, K562, HepG2) and embryonic stem cells. Results should be interpreted as hypothesis-generating observations, not as evidence of immune cell-specific TF binding.

In [ ]:
# ChEA 2022 TF enrichment
tf_enr = gp.enrichr(
    gene_list=gene_list,
    gene_sets='ChEA_2022',
    organism='human',
    outdir=None
)

tf_results = tf_enr.results
tf_results['FDR'] = tf_results['Adjusted P-value']
tf_results = tf_results.sort_values('FDR')

# Significant at different thresholds
sig_01 = tf_results[tf_results['FDR'] < 0.01]
sig_05 = tf_results[tf_results['FDR'] < 0.05]
sig_10 = tf_results[tf_results['FDR'] < 0.10]

print('=== ChEA 2022 TF Enrichment ===')
print(f'Significant at FDR < 0.01: {len(sig_01)} TFs')
print(f'Significant at FDR < 0.05: {len(sig_05)} TFs')
print(f'Significant at FDR < 0.10: {len(sig_10)} TFs')

print(f'\nTop 15 TFs (sorted by FDR):')
display_cols = ['Term', 'Overlap', 'P-value', 'FDR', 'Odds Ratio']
print(tf_results[display_cols].head(15).to_string(index=False))

# Save
tf_results.to_csv(f'{OUTPUT_DIR}/ChEA2022_TF_enrichment_SIEM_20genes.csv', index=False)
print(f'\nSaved: ChEA2022_TF_enrichment_SIEM_20genes.csv')

## 4.3 Generate TF Barplot Figure

In [ ]:
# Prepare data for plotting
tf_top = tf_results.head(15).copy()
tf_top['TF'] = tf_top['Term'].str.replace('\s+human.*$', '', regex=True)
tf_top['neg_log_FDR'] = -np.log10(tf_top['FDR'])

# Color by significance level
def color_by_fdr(fdr):
    if fdr < 0.01: return '#C0392B'
    elif fdr < 0.05: return '#E67E22'
    else: return '#3498DB'

tf_top['color'] = tf_top['FDR'].apply(color_by_fdr)

# Plot
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.barh(range(len(tf_top)), tf_top['neg_log_FDR'], color=tf_top['color'],
               edgecolor='white', linewidth=0.5, height=0.7)
ax.set_yticks(range(len(tf_top)))
ax.set_yticklabels(tf_top['TF'], fontsize=8)
ax.invert_yaxis()

# Significance thresholds
ax.axvline(x=-np.log10(0.01), color='#C0392B', linestyle='--', linewidth=1, alpha=0.7, label='FDR = 0.01')
ax.axvline(x=-np.log10(0.05), color='gray', linestyle='--', linewidth=1, alpha=0.5, label='FDR = 0.05')

# Value labels
for i, (bar, val) in enumerate(zip(bars, tf_top['neg_log_FDR'])):
    ax.text(val + 0.05, i, f'{val:.2f}', va='center', fontsize=7, color='#333')

ax.set_xlabel('-log$_{10}$(FDR)', fontsize=9)
ax.set_title('Transcription Factor Enrichment (ChEA 2022)', fontsize=11, fontweight='bold', pad=10)
ax.legend(fontsize=8, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, max(tf_top['neg_log_FDR']) + 0.5)

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/Fig_v6_3_tf.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: Fig_v6_3_tf.png')

## 4.4 Summary Table for Manuscript

In [ ]:
# Create combined summary table
# GO results (FDR < 0.05)
go_sig = go_results[go_results['Adjusted P-value'] < 0.05].copy()
go_sig['Category'] = 'GO BP'
go_sig['Term_clean'] = go_sig['Term'].str.replace(' \(GO:.*\)', '', regex=True)

# KEGG results (FDR < 0.25)
kegg_sig = kegg_results[kegg_results['Adjusted P-value'] < 0.25].copy()
kegg_sig['Category'] = 'KEGG'
kegg_sig['Term_clean'] = kegg_sig['Term']

# Combine
summary = pd.concat([
    go_sig[['Category', 'Term_clean', 'Adjusted P-value', 'Odds Ratio', 'Genes']].rename(
        columns={'Term_clean': 'Term', 'Adjusted P-value': 'FDR'}),
    kegg_sig[['Category', 'Term_clean', 'Adjusted P-value', 'Odds Ratio', 'Genes']].rename(
        columns={'Term_clean': 'Term', 'Adjusted P-value': 'FDR'})
])

summary = summary.sort_values('FDR')
print('=== Combined Enrichment Summary (Manuscript Table) ===')
print(summary.head(20).to_string(index=False))

summary.to_csv(f'{OUTPUT_DIR}/combined_enrichment_summary.csv', index=False)
print(f'\nSaved: combined_enrichment_summary.csv')